In [12]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pymongo import MongoClient
from dotenv import load_dotenv

# Configurações visuais
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Carregar variáveis de ambiente
load_dotenv()

# Conexão MongoDB
MONGO_URI = os.getenv("MONGODB_URI")
mongo_client = MongoClient(MONGO_URI)
db = mongo_client["experiments_db"]
llm_judge_collection = db["llm_judge_comparative_metrics"]

# Métricas avaliadas
METRICS = ["coherence", "specificity", "informativeness", "relevance", "Understandability"]
METRIC_LABELS = {
    "coherence": "Coerência",
    "specificity": "Especificidade",
    "informativeness": "Informatividade",
    "relevance": "Relevância",
    "Understandability": "Compreensibilidade"
}

print("✅ Bibliotecas importadas e conexão MongoDB estabelecida!")

✅ Bibliotecas importadas e conexão MongoDB estabelecida!


## 2. Carregamento e Processamento dos Dados

In [13]:
# Carregar todos os documentos da collection
documents = list(llm_judge_collection.find())

print(f"📊 Total de experimentos encontrados: {len(documents)}\n")

# Processar dados
models = []

for doc in documents:
    experiment_name = doc["experiment_name"]
    # Extrair nome do modelo (remover prefixo "LLM_Judge_")
    model_name = experiment_name.replace("LLM_Judge_", "")
    models.append(model_name)
    
print("✅ Dados processados com sucesso!")
print(f"\n📋 Modelos encontrados:")
for model in models:
    print(f"   • {model}")

📊 Total de experimentos encontrados: 4

✅ Dados processados com sucesso!

📋 Modelos encontrados:
   • gpt-4o-mini
   • llama-3.3-70b-versatile
   • gpt-3.5-turbo-0125
   • llama2:7b


## 3. Tabela Detalhada - Abordagem MTI

In [14]:
# Criar tabela detalhada para MTI
mti_table_rows = []

ORDERED_MODELS = [
    "llama2:7b",
    "gpt-3.5-turbo-0125",
    "llama-3.3-70b-versatile",
    "gpt-4o-mini"
]

for doc in documents:
    experiment_name = doc["experiment_name"]
    model_name = experiment_name.replace("LLM_Judge_", "")
    
    row_data = {"Modelo": model_name}
    
    # MTI metrics
    mti_metrics = doc.get("MTI_metrics", {})
    
    # Para cada métrica individual
    for metric in METRICS:
        if metric in mti_metrics:
            mean_val = mti_metrics[metric].get("mean", 0)
            std_val = mti_metrics[metric].get("std", 0)
            count_max = mti_metrics[metric].get("count_max_score", 0)
            total = mti_metrics[metric].get("total_samples", 1)
            
            pct_max = (count_max / total * 100) if total > 0 else 0
            
            col_value = f"μ: {mean_val:.2f} | σ: {std_val:.2f} | Max: {pct_max:.2f}%"
        else:
            col_value = "N/A"
        
        metric_label = METRIC_LABELS[metric]
        row_data[metric_label] = col_value
    
    # Média geral
    overall = doc.get("overall_statistics", {}).get("MTI", {})
    overall_mean = overall.get("mean", 0)
    overall_std = overall.get("std", 0)
    overall_count_max = overall.get("count_max_score", 0)
    overall_total = overall.get("total_samples", 1)
    overall_pct_max = (overall_count_max / overall_total * 100) if overall_total > 0 else 0
    
    row_data["Média"] = f"μ: {overall_mean:.2f} | σ: {overall_std:.2f} | Max: {overall_pct_max:.2f}%"
    
    mti_table_rows.append(row_data)

df_mti_detailed = pd.DataFrame(mti_table_rows)

# 🔥 Ordenar pela lista desejada
df_mti_detailed["sort_key"] = df_mti_detailed["Modelo"].apply(
    lambda x: ORDERED_MODELS.index(x) if x in ORDERED_MODELS else 999
)

df_mti_detailed = df_mti_detailed.sort_values("sort_key").drop(columns=["sort_key"]).reset_index(drop=True)

print("\n" + "="*150)
print("📊 TABELA DETALHADA - ABORDAGEM MTI")
print("="*150)
print("Formato: μ: [Média] | σ: [Desvio Padrão] | Max: [% Pontuação Máxima]")
print("-"*150)
display(df_mti_detailed)



📊 TABELA DETALHADA - ABORDAGEM MTI
Formato: μ: [Média] | σ: [Desvio Padrão] | Max: [% Pontuação Máxima]
------------------------------------------------------------------------------------------------------------------------------------------------------


,Modelo,Coerência,Especificidade,Informatividade,Relevância,Compreensibilidade,Média
0,llama2:7b,μ: 3.84 | σ: 0.71 | Max: 14.92%,μ: 3.59 | σ: 0.95 | Max: 11.83%,μ: 3.82 | σ: 0.78 | Max: 17.29%,μ: 4.56 | σ: 0.74 | Max: 67.79%,μ: 3.96 | σ: 0.78 | Max: 25.04%,μ: 3.95 | σ: 0.86 | Max: 27.38%
1,gpt-3.5-turbo-0125,μ: 4.34 | σ: 0.74 | Max: 45.88%,μ: 3.96 | σ: 1.15 | Max: 35.79%,μ: 4.14 | σ: 0.89 | Max: 39.38%,μ: 4.79 | σ: 0.60 | Max: 84.96%,μ: 4.47 | σ: 0.71 | Max: 55.50%,μ: 4.34 | σ: 0.89 | Max: 52.30%
2,llama-3.3-70b-versatile,μ: 4.52 | σ: 0.53 | Max: 53.87%,μ: 4.28 | σ: 0.94 | Max: 44.50%,μ: 4.46 | σ: 0.57 | Max: 49.54%,μ: 4.96 | σ: 0.21 | Max: 95.67%,μ: 4.65 | σ: 0.49 | Max: 65.96%,μ: 4.57 | σ: 0.64 | Max: 61.91%
3,gpt-4o-mini,μ: 4.63 | σ: 0.50 | Max: 63.92%,μ: 4.37 | σ: 0.96 | Max: 53.08%,μ: 4.55 | σ: 0.55 | Max: 57.67%,μ: 4.96 | σ: 0.20 | Max: 95.96%,μ: 4.71 | σ: 0.47 | Max: 71.67%,μ: 4.64 | σ: 0.62 | Max: 68.46%


## 4. Tabela Detalhada - Abordagem STI

In [15]:
# Criar tabela detalhada para STI
sti_table_rows = []

ORDERED_MODELS = [
    "llama2:7b",
    "gpt-3.5-turbo-0125",
    "llama-3.3-70b-versatile",
    "gpt-4o-mini"
]

for doc in documents:
    experiment_name = doc["experiment_name"]
    model_name = experiment_name.replace("LLM_Judge_", "")
    
    row_data = {"Modelo": model_name}
    
    # STI metrics
    sti_metrics = doc.get("STI_metrics", {})
    
    for metric in METRICS:
        if metric in sti_metrics:
            mean_val = sti_metrics[metric].get("mean", 0)
            std_val = sti_metrics[metric].get("std", 0)
            count_max = sti_metrics[metric].get("count_max_score", 0)
            total = sti_metrics[metric].get("total_samples", 1)
            
            pct_max = (count_max / total * 100) if total > 0 else 0
            
            col_value = f"μ: {mean_val:.2f} | σ: {std_val:.2f} | Max: {pct_max:.2f}%"
        else:
            col_value = "N/A"
        
        metric_label = METRIC_LABELS[metric]
        row_data[metric_label] = col_value
    
    # Média geral
    overall = doc.get("overall_statistics", {}).get("STI", {})
    overall_mean = overall.get("mean", 0)
    overall_std = overall.get("std", 0)
    overall_count_max = overall.get("count_max_score", 0)
    overall_total = overall.get("total_samples", 1)
    overall_pct_max = (overall_count_max / overall_total * 100) if overall_total > 0 else 0
    
    row_data["Média"] = f"μ: {overall_mean:.2f} | σ: {overall_std:.2f} | Max: {overall_pct_max:.2f}%"
    
    sti_table_rows.append(row_data)

df_sti_detailed = pd.DataFrame(sti_table_rows)

# 🔥 Ordenar pela lista desejada
df_sti_detailed["sort_key"] = df_sti_detailed["Modelo"].apply(
    lambda x: ORDERED_MODELS.index(x) if x in ORDERED_MODELS else 999
)

df_sti_detailed = df_sti_detailed.sort_values("sort_key").drop(columns=["sort_key"]).reset_index(drop=True)

print("\n" + "="*150)
print("📊 TABELA DETALHADA - ABORDAGEM STI")
print("="*150)
print("Formato: μ: [Média] | σ: [Desvio Padrão] | Max: [% Pontuação Máxima]")
print("-"*150)
display(df_sti_detailed)



📊 TABELA DETALHADA - ABORDAGEM STI
Formato: μ: [Média] | σ: [Desvio Padrão] | Max: [% Pontuação Máxima]
------------------------------------------------------------------------------------------------------------------------------------------------------


,Modelo,Coerência,Especificidade,Informatividade,Relevância,Compreensibilidade,Média
0,llama2:7b,μ: 3.56 | σ: 0.76 | Max: 7.92%,μ: 3.31 | σ: 0.98 | Max: 6.67%,μ: 3.55 | σ: 0.85 | Max: 10.58%,μ: 4.28 | σ: 0.90 | Max: 51.33%,μ: 3.68 | σ: 0.82 | Max: 15.04%,μ: 3.68 | σ: 0.92 | Max: 18.31%
1,gpt-3.5-turbo-0125,μ: 4.40 | σ: 0.63 | Max: 47.33%,μ: 4.08 | σ: 1.07 | Max: 37.50%,μ: 4.29 | σ: 0.74 | Max: 42.71%,μ: 4.87 | σ: 0.40 | Max: 89.12%,μ: 4.53 | σ: 0.59 | Max: 57.42%,μ: 4.43 | σ: 0.77 | Max: 54.82%
2,llama-3.3-70b-versatile,μ: 4.41 | σ: 0.58 | Max: 45.88%,μ: 4.19 | σ: 0.94 | Max: 37.83%,μ: 4.37 | σ: 0.59 | Max: 42.42%,μ: 4.91 | σ: 0.34 | Max: 91.83%,μ: 4.55 | σ: 0.55 | Max: 57.42%,μ: 4.49 | σ: 0.67 | Max: 55.07%
3,gpt-4o-mini,μ: 4.57 | σ: 0.52 | Max: 57.71%,μ: 4.33 | σ: 0.95 | Max: 48.21%,μ: 4.51 | σ: 0.54 | Max: 53.17%,μ: 4.96 | σ: 0.19 | Max: 96.46%,μ: 4.64 | σ: 0.50 | Max: 65.21%,μ: 4.60 | σ: 0.62 | Max: 64.15%


## 5. Exportação das Tabelas para LaTeX e PNG

In [16]:
import dataframe_image as dfi
from datetime import datetime

# Criar diretório para exportação se não existir
export_dir = "inference/llm_judge_result/exports"
os.makedirs(export_dir, exist_ok=True)

# Timestamp para nomes de arquivo únicos
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

print("🚀 Iniciando exportação das tabelas...")
print("="*100)

🚀 Iniciando exportação das tabelas...


### 5.1 Exportar Tabela MTI

In [ ]:
# Exportar Tabela MTI para LaTeX
mti_latex_file = f"{export_dir}/tabela_mti_llm_judge.tex"
mti_png_file = f"{export_dir}/tabela_mti_llm_judge.png"

# Configurar estilo para melhor visualização
df_mti_styled = df_mti_detailed.style.set_properties(**{
    'text-align': 'center',
    'font-size': '10pt',
    'border': '1px solid black'
}).set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#4472C4'), ('color', 'white'), 
                                   ('font-weight', 'bold'), ('text-align', 'center'),
                                   ('border', '1px solid black')]},
    {'selector': 'td', 'props': [('border', '1px solid black')]},
    {'selector': 'table', 'props': [('border-collapse', 'collapse'), ('width', '100%')]}
])

# Função para gerar LaTeX customizado no formato makecell
def generate_custom_latex(df, caption, label, approach_name):
    """Gera código LaTeX formatado com makecell (formato vertical)"""
    
    # Cabeçalho do LaTeX
    latex_lines = []
    latex_lines.append("\\begin{table}[htbp]")
    latex_lines.append(f"\\caption{{{caption}}}")
    latex_lines.append(f"\\label{{{label}}}")
    latex_lines.append("\\centering")
    latex_lines.append("")
    latex_lines.append("\\renewcommand{\\arraystretch}{1.45} % aumenta o espaçamento entre linhas")
    latex_lines.append("")
    latex_lines.append("\\begin{adjustbox}{max width=\\textwidth}")
    
    # Determinar número de colunas
    num_cols = len(df.columns)
    col_format = 'c' * num_cols
    
    latex_lines.append(f"\\begin{{tabular}}{{{col_format}}}")
    latex_lines.append("\\toprule")
    
    # Cabeçalho da tabela
    header_parts = []
    for col in df.columns:
        if col == "Modelo":
            header_parts.append("\\textbf{Modelo}")
        elif col == "Média":
            header_parts.append("\\textbf{Média}")
        else:
            # Traduzir nomes das métricas
            header_parts.append(f"\\textbf{{{col}}}")
    
    latex_lines.append(" & ".join(header_parts) + " \\\\")
    latex_lines.append("\\midrule")
    
    # Linhas de dados
    for idx, row in df.iterrows():
        row_parts = []
        for col in df.columns:
            cell_value = row[col]
            
            if col == "Modelo":
                # Nome do modelo com makecell
                row_parts.append(f"\\makecell{{{str(cell_value)}}}")
            else:
                # Processar valores de métrica (formato: "μ: 4.57 | σ: 0.52 | Max: 57.71%")
                cell_str = str(cell_value)
                
                if "N/A" in cell_str:
                    row_parts.append("\\makecell[c]{N/A}")
                else:
                    # Parsear o formato "μ: 4.57 | σ: 0.52 | Max: 57.71%"
                    if "|" in cell_str:
                        parts = cell_str.split("|")
                        
                        if len(parts) == 3:
                            # Formato completo com média, desvio padrão e max
                            mean_part = parts[0].strip()  # "μ: 4.57"
                            std_part = parts[1].strip()   # "σ: 0.52"
                            max_part = parts[2].strip()   # "Max: 57.71%"
                            
                            # Extrair valores numéricos
                            mean_value = mean_part.replace("μ:", "").strip()
                            std_value = std_part.replace("σ:", "").strip()
                            max_value = max_part.replace("Max:", "").replace("%", "").strip()
                            
                            # Formatar no padrão LaTeX com makecell (vertical)
                            latex_cell = f"\\makecell[c]{{Max: {max_value}\\% \\\\ $\\mu$: {mean_value} \\\\ $\\sigma$: {std_value}}}"
                            row_parts.append(latex_cell)
                        else:
                            # Formato antigo (apenas mean e max) - manter compatibilidade
                            mean_part = parts[0].strip()
                            max_part = parts[1].strip()
                            
                            mean_value = mean_part.replace("μ:", "").strip()
                            max_value = max_part.replace("Max:", "").replace("%", "").strip()
                            
                            latex_cell = f"\\makecell[c]{{Max: {max_value}\\% \\\\ $\\mu$: {mean_value}}}"
                            row_parts.append(latex_cell)
                    else:
                        row_parts.append(f"\\makecell[c]{{{cell_str}}}")
        
        latex_lines.append(" &\n".join(row_parts) + " \\\\")
        
        # Adicionar midrule entre linhas (exceto após a última)
        if idx < len(df) - 1:
            latex_lines.append("\\midrule")
            latex_lines.append("")
    
    # Rodapé da tabela
    latex_lines.append("\\bottomrule")
    latex_lines.append("")
    latex_lines.append("\\end{tabular}")
    latex_lines.append("\\end{adjustbox}")
    latex_lines.append("")
    latex_lines.append("\\end{table}")
    
    return "\n".join(latex_lines)

# Gerar LaTeX customizado para MTI
latex_output = generate_custom_latex(
    df_mti_detailed,
    caption="Resultados Detalhados da Abordagem MTI - Métricas LLM Judge por Modelo",
    label="tab:mti_llm_judge",
    approach_name="MTI"
)

# Salvar LaTeX
with open(mti_latex_file, 'w', encoding='utf-8') as f:
    f.write(latex_output)

# Exportar para PNG
try:
    dfi.export(df_mti_styled, mti_png_file, max_rows=-1, max_cols=-1)
    print(f"✅ Tabela MTI exportada com sucesso!")
except Exception as e:
    # Alternativa usando matplotlib
    print(f"⚠️ Usando método alternativo para PNG: {str(e)}")
    
    fig, ax = plt.subplots(figsize=(20, len(df_mti_detailed) * 0.5 + 2))
    ax.axis('tight')
    ax.axis('off')
    
    table = ax.table(cellText=df_mti_detailed.values, 
                     colLabels=df_mti_detailed.columns,
                     cellLoc='center', 
                     loc='center',
                     bbox=[0, 0, 1, 1])
    
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 2)
    
    # Estilizar cabeçalho
    for i in range(len(df_mti_detailed.columns)):
        table[(0, i)].set_facecolor('#4472C4')
        table[(0, i)].set_text_props(weight='bold', color='white')
    
    # Estilizar células
    for i in range(1, len(df_mti_detailed) + 1):
        for j in range(len(df_mti_detailed.columns)):
            if i % 2 == 0:
                table[(i, j)].set_facecolor('#E7E6E6')
    
    plt.savefig(mti_png_file, dpi=300, bbox_inches='tight', pad_inches=0.2)
    plt.close()
    print(f"✅ Tabela MTI exportada com sucesso (método alternativo)!")

print(f"   📄 LaTeX: {mti_latex_file}")
print(f"   🖼️  PNG: {mti_png_file}")
print("-"*100)

⚠️ Usando método alternativo para PNG: It looks like you are using Playwright Sync API inside the asyncio loop.
Please use the Async API instead.
✅ Tabela MTI exportada com sucesso (método alternativo)!
   📄 LaTeX: inference/llm_judge_result/exports/tabela_mti_llm_judge.tex
   🖼️  PNG: inference/llm_judge_result/exports/tabela_mti_llm_judge.png
----------------------------------------------------------------------------------------------------
✅ Tabela MTI exportada com sucesso (método alternativo)!
   📄 LaTeX: inference/llm_judge_result/exports/tabela_mti_llm_judge.tex
   🖼️  PNG: inference/llm_judge_result/exports/tabela_mti_llm_judge.png
----------------------------------------------------------------------------------------------------


### 5.2 Exportar Tabela STI

In [ ]:
# Exportar Tabela STI para LaTeX
sti_latex_file = f"{export_dir}/tabela_sti_llm_judge.tex"
sti_png_file = f"{export_dir}/tabela_sti_llm_judge.png"

# Configurar estilo
df_sti_styled = df_sti_detailed.style.set_properties(**{
    'text-align': 'center',
    'font-size': '10pt',
    'border': '1px solid black'
}).set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#1A76FF'), ('color', 'white'), 
                                   ('font-weight', 'bold'), ('text-align', 'center'),
                                   ('border', '1px solid black')]},
    {'selector': 'td', 'props': [('border', '1px solid black')]},
    {'selector': 'table', 'props': [('border-collapse', 'collapse'), ('width', '100%')]}
])

# Gerar LaTeX customizado para STI
latex_output = generate_custom_latex(
    df_sti_detailed,
    caption="Resultados Detalhados da Abordagem STI - Métricas LLM Judge por Modelo. Porcentagem de Nota Máxima (Max); Média ($\\mu$); Desvio Padrão ($\\sigma$)",
    label="tab:sti_llm_judge",
    approach_name="STI"
)

# Salvar LaTeX
with open(sti_latex_file, 'w', encoding='utf-8') as f:
    f.write(latex_output)

# Exportar para PNG
try:
    dfi.export(df_sti_styled, sti_png_file, max_rows=-1, max_cols=-1)
    print(f"✅ Tabela STI exportada com sucesso!")
except Exception as e:
    # Alternativa usando matplotlib
    print(f"⚠️ Usando método alternativo para PNG: {str(e)}")
    
    fig, ax = plt.subplots(figsize=(20, len(df_sti_detailed) * 0.5 + 2))
    ax.axis('tight')
    ax.axis('off')
    
    table = ax.table(cellText=df_sti_detailed.values, 
                     colLabels=df_sti_detailed.columns,
                     cellLoc='center', 
                     loc='center',
                     bbox=[0, 0, 1, 1])
    
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 2)
    
    # Estilizar cabeçalho
    for i in range(len(df_sti_detailed.columns)):
        table[(0, i)].set_facecolor('#1A76FF')
        table[(0, i)].set_text_props(weight='bold', color='white')
    
    # Estilizar células
    for i in range(1, len(df_sti_detailed) + 1):
        for j in range(len(df_sti_detailed.columns)):
            if i % 2 == 0:
                table[(i, j)].set_facecolor('#E7E6E6')
    
    plt.savefig(sti_png_file, dpi=300, bbox_inches='tight', pad_inches=0.2)
    plt.close()
    print(f"✅ Tabela STI exportada com sucesso (método alternativo)!")

print(f"   📄 LaTeX: {sti_latex_file}")
print(f"   🖼️  PNG: {sti_png_file}")
print("-"*100)

⚠️ Usando método alternativo para PNG: It looks like you are using Playwright Sync API inside the asyncio loop.
Please use the Async API instead.
✅ Tabela STI exportada com sucesso (método alternativo)!
   📄 LaTeX: inference/llm_judge_result/exports/tabela_sti_llm_judge.tex
   🖼️  PNG: inference/llm_judge_result/exports/tabela_sti_llm_judge.png
----------------------------------------------------------------------------------------------------
✅ Tabela STI exportada com sucesso (método alternativo)!
   📄 LaTeX: inference/llm_judge_result/exports/tabela_sti_llm_judge.tex
   🖼️  PNG: inference/llm_judge_result/exports/tabela_sti_llm_judge.png
----------------------------------------------------------------------------------------------------


### 5.5 Resumo da Exportação

### 5.3 Gerar Tabela de Diferença (Δ = MTI - STI)

In [17]:
# Criar tabela de diferença (MTI - STI)
diff_table_rows = []

ORDERED_MODELS = [
    "llama2:7b",
    "gpt-3.5-turbo-0125",
    "llama-3.3-70b-versatile",
    "gpt-4o-mini"
]

for doc in documents:
    experiment_name = doc["experiment_name"]
    model_name = experiment_name.replace("LLM_Judge_", "")
    
    row_data = {"Modelo": model_name}
    
    # MTI e STI metrics
    mti_metrics = doc.get("MTI_metrics", {})
    sti_metrics = doc.get("STI_metrics", {})
    
    # Para cada métrica individual
    for metric in METRICS:
        if metric in mti_metrics and metric in sti_metrics:
            # MTI values
            mti_mean = mti_metrics[metric].get("mean", 0)
            mti_std = mti_metrics[metric].get("std", 0)
            mti_count_max = mti_metrics[metric].get("count_max_score", 0)
            mti_total = mti_metrics[metric].get("total_samples", 1)
            mti_pct_max = (mti_count_max / mti_total * 100) if mti_total > 0 else 0
            
            # STI values
            sti_mean = sti_metrics[metric].get("mean", 0)
            sti_std = sti_metrics[metric].get("std", 0)
            sti_count_max = sti_metrics[metric].get("count_max_score", 0)
            sti_total = sti_metrics[metric].get("total_samples", 1)
            sti_pct_max = (sti_count_max / sti_total * 100) if sti_total > 0 else 0
            
            # Calcular diferenças
            diff_max = mti_pct_max - sti_pct_max
            diff_mean = mti_mean - sti_mean
            diff_std = mti_std - sti_std
            
            # Formatar com cores (para display Python)
            col_value = f"ΔMax: {diff_max:+.2f}% | Δμ: {diff_mean:+.2f} | Δσ: {diff_std:+.2f}"
        else:
            col_value = "N/A"
        
        metric_label = METRIC_LABELS[metric]
        row_data[metric_label] = col_value
    
    # Diferença na média geral
    mti_overall = doc.get("overall_statistics", {}).get("MTI", {})
    sti_overall = doc.get("overall_statistics", {}).get("STI", {})
    
    mti_overall_mean = mti_overall.get("mean", 0)
    mti_overall_std = mti_overall.get("std", 0)
    mti_overall_count_max = mti_overall.get("count_max_score", 0)
    mti_overall_total = mti_overall.get("total_samples", 1)
    mti_overall_pct_max = (mti_overall_count_max / mti_overall_total * 100) if mti_overall_total > 0 else 0
    
    sti_overall_mean = sti_overall.get("mean", 0)
    sti_overall_std = sti_overall.get("std", 0)
    sti_overall_count_max = sti_overall.get("count_max_score", 0)
    sti_overall_total = sti_overall.get("total_samples", 1)
    sti_overall_pct_max = (sti_overall_count_max / sti_overall_total * 100) if sti_overall_total > 0 else 0
    
    diff_overall_max = mti_overall_pct_max - sti_overall_pct_max
    diff_overall_mean = mti_overall_mean - sti_overall_mean
    diff_overall_std = mti_overall_std - sti_overall_std
    
    row_data["Média"] = f"ΔMax: {diff_overall_max:+.2f}% | Δμ: {diff_overall_mean:+.2f} | Δσ: {diff_overall_std:+.2f}"
    
    diff_table_rows.append(row_data)

df_diff_detailed = pd.DataFrame(diff_table_rows)

# Ordenar pela lista desejada
df_diff_detailed["sort_key"] = df_diff_detailed["Modelo"].apply(
    lambda x: ORDERED_MODELS.index(x) if x in ORDERED_MODELS else 999
)

df_diff_detailed = df_diff_detailed.sort_values("sort_key").drop(columns=["sort_key"]).reset_index(drop=True)

print("\n" + "="*150)
print("📊 TABELA DE DIFERENÇA - MTI vs STI (Δ = MTI - STI)")
print("="*150)
print("Formato: ΔMax: [Diferença %] | Δμ: [Diferença Média] | Δσ: [Diferença Desvio Padrão]")
print("Valores positivos (+) indicam que MTI teve melhor desempenho")
print("Valores negativos (-) indicam que STI teve melhor desempenho")
print("-"*150)
display(df_diff_detailed)


📊 TABELA DE DIFERENÇA - MTI vs STI (Δ = MTI - STI)
Formato: ΔMax: [Diferença %] | Δμ: [Diferença Média] | Δσ: [Diferença Desvio Padrão]
Valores positivos (+) indicam que MTI teve melhor desempenho
Valores negativos (-) indicam que STI teve melhor desempenho
------------------------------------------------------------------------------------------------------------------------------------------------------


,Modelo,Coerência,Especificidade,Informatividade,Relevância,Compreensibilidade,Média
0,llama2:7b,ΔMax: +7.00% | Δμ: +0.28 | Δσ: -0.05,ΔMax: +5.17% | Δμ: +0.28 | Δσ: -0.03,ΔMax: +6.71% | Δμ: +0.27 | Δσ: -0.07,ΔMax: +16.46% | Δμ: +0.28 | Δσ: -0.16,ΔMax: +10.00% | Δμ: +0.28 | Δσ: -0.04,ΔMax: +9.07% | Δμ: +0.27 | Δσ: -0.06
1,gpt-3.5-turbo-0125,ΔMax: -1.46% | Δμ: -0.06 | Δσ: +0.11,ΔMax: -1.71% | Δμ: -0.12 | Δσ: +0.08,ΔMax: -3.33% | Δμ: -0.15 | Δσ: +0.15,ΔMax: -4.17% | Δμ: -0.08 | Δσ: +0.20,ΔMax: -1.92% | Δμ: -0.06 | Δσ: +0.12,ΔMax: -2.52% | Δμ: -0.09 | Δσ: +0.12
2,llama-3.3-70b-versatile,ΔMax: +8.00% | Δμ: +0.11 | Δσ: -0.05,ΔMax: +6.67% | Δμ: +0.09 | Δσ: +0.00,ΔMax: +7.12% | Δμ: +0.09 | Δσ: -0.02,ΔMax: +3.83% | Δμ: +0.05 | Δσ: -0.13,ΔMax: +8.54% | Δμ: +0.10 | Δσ: -0.06,ΔMax: +6.83% | Δμ: +0.08 | Δσ: -0.03
3,gpt-4o-mini,ΔMax: +6.21% | Δμ: +0.06 | Δσ: -0.02,ΔMax: +4.88% | Δμ: +0.04 | Δσ: +0.01,ΔMax: +4.50% | Δμ: +0.04 | Δσ: +0.01,ΔMax: -0.50% | Δμ: +0.00 | Δσ: +0.01,ΔMax: +6.46% | Δμ: +0.07 | Δσ: -0.03,ΔMax: +4.31% | Δμ: +0.04 | Δσ: +0.00


### 5.4 Exportar Tabela de Diferença para LaTeX e PNG

In [18]:
# Exportar Tabela de Diferença para LaTeX
diff_latex_file = f"{export_dir}/tabela_diff_mti_sti_llm_judge.tex"
diff_png_file = f"{export_dir}/tabela_diff_mti_sti_llm_judge.png"

# Função para gerar LaTeX da tabela de diferença com cores
def generate_diff_latex(df, caption, label):
    """Gera código LaTeX formatado para tabela de diferença com cores"""
    
    # Cabeçalho do LaTeX
    latex_lines = []
    latex_lines.append("\\begin{table}[htbp]")
    latex_lines.append(f"\\caption{{{caption}}}")
    latex_lines.append(f"\\label{{{label}}}")
    latex_lines.append("\\centering")
    latex_lines.append("")
    latex_lines.append("\\renewcommand{\\arraystretch}{1.45} % aumenta o espaçamento entre linhas")
    latex_lines.append("")
    latex_lines.append("\\begin{adjustbox}{max width=\\textwidth}")
    
    # Determinar número de colunas
    num_cols = len(df.columns)
    col_format = 'c' * num_cols
    
    latex_lines.append(f"\\begin{{tabular}}{{{col_format}}}")
    latex_lines.append("\\toprule")
    
    # Cabeçalho da tabela
    header_parts = []
    for col in df.columns:
        if col == "Modelo":
            header_parts.append("\\textbf{Modelo}")
        elif col == "Média":
            header_parts.append("\\textbf{Média}")
        else:
            header_parts.append(f"\\textbf{{{col}}}")
    
    latex_lines.append(" & ".join(header_parts) + " \\\\")
    latex_lines.append("\\midrule")
    
    # Linhas de dados
    for idx, row in df.iterrows():
        row_parts = []
        for col in df.columns:
            cell_value = row[col]
            
            if col == "Modelo":
                # Nome do modelo com makecell
                row_parts.append(f"\\makecell{{{str(cell_value)}}}")
            else:
                # Processar valores de diferença (formato: "ΔMax: +7.00% | Δμ: +0.28 | Δσ: -0.05")
                cell_str = str(cell_value)
                
                if "N/A" in cell_str:
                    row_parts.append("\\makecell[c]{N/A}")
                else:
                    # Parsear o formato "ΔMax: +7.00% | Δμ: +0.28 | Δσ: -0.05"
                    if "|" in cell_str:
                        parts = cell_str.split("|")
                        
                        if len(parts) == 3:
                            max_part = parts[0].strip()  # "ΔMax: +7.00%"
                            mean_part = parts[1].strip()  # "Δμ: +0.28"
                            std_part = parts[2].strip()   # "Δσ: -0.05"
                            
                            # Extrair valores numéricos
                            max_value_str = max_part.replace("ΔMax:", "").replace("%", "").strip()
                            mean_value_str = mean_part.replace("Δμ:", "").strip()
                            std_value_str = std_part.replace("Δσ:", "").strip()
                            
                            # Converter para float para verificar sinal
                            try:
                                max_val = float(max_value_str)
                                mean_val = float(mean_value_str)
                                std_val = float(std_value_str)
                                
                                # Determinar cores baseadas no sinal
                                max_color = "green" if max_val > 0 else "red" if max_val < 0 else "black"
                                mean_color = "green" if mean_val > 0 else "red" if mean_val < 0 else "black"
                                std_color = "green" if std_val > 0 else "red" if std_val < 0 else "black"
                                
                                # Formatar com cores no LaTeX (formato vertical com makecell)
                                latex_cell = f"\\makecell[c]{{\\textcolor{{{max_color}}}{{{max_value_str}\\%}} \\\\ \\textcolor{{{mean_color}}}{{$\\Delta\\mu$: {mean_value_str}}} \\\\ \\textcolor{{{std_color}}}{{$\\Delta\\sigma$: {std_value_str}}}}}"
                                row_parts.append(latex_cell)
                            except ValueError:
                                # Se não conseguir converter, usar sem cor
                                latex_cell = f"\\makecell[c]{{{max_value_str}\\% \\\\ $\\Delta\\mu$: {mean_value_str} \\\\ $\\Delta\\sigma$: {std_value_str}}}"
                                row_parts.append(latex_cell)
                        else:
                            row_parts.append(f"\\makecell[c]{{{cell_str}}}")
                    else:
                        row_parts.append(f"\\makecell[c]{{{cell_str}}}")
        
        latex_lines.append(" &\n".join(row_parts) + " \\\\")
        
        # Adicionar midrule entre linhas (exceto após a última)
        if idx < len(df) - 1:
            latex_lines.append("\\midrule")
            latex_lines.append("")
    
    # Rodapé da tabela
    latex_lines.append("\\bottomrule")
    latex_lines.append("")
    latex_lines.append("\\end{tabular}")
    latex_lines.append("\\end{adjustbox}")
    latex_lines.append("")
    latex_lines.append("\\end{table}")
    
    return "\n".join(latex_lines)

# Gerar LaTeX customizado para tabela de diferença
latex_output = generate_diff_latex(
    df_diff_detailed,
    caption="Diferença de Desempenho: MTI vs STI ($\\Delta$ = MTI - STI). Valores positivos (verde) indicam superioridade da abordagem MTI. Valores negativos (vermelho) indicam superioridade da abordagem STI.",
    label="tab:diff_mti_sti_llm_judge"
)

# Salvar LaTeX
with open(diff_latex_file, 'w', encoding='utf-8') as f:
    f.write(latex_output)

# Configurar estilo para PNG
df_diff_styled = df_diff_detailed.style.set_properties(**{
    'text-align': 'center',
    'font-size': '10pt',
    'border': '1px solid black'
}).set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#FF8C00'), ('color', 'white'), 
                                   ('font-weight', 'bold'), ('text-align', 'center'),
                                   ('border', '1px solid black')]},
    {'selector': 'td', 'props': [('border', '1px solid black')]},
    {'selector': 'table', 'props': [('border-collapse', 'collapse'), ('width', '100%')]}
])

# Exportar para PNG
try:
    dfi.export(df_diff_styled, diff_png_file, max_rows=-1, max_cols=-1)
    print(f"✅ Tabela de Diferença exportada com sucesso!")
except Exception as e:
    # Alternativa usando matplotlib
    print(f"⚠️ Usando método alternativo para PNG: {str(e)}")
    
    fig, ax = plt.subplots(figsize=(20, len(df_diff_detailed) * 0.5 + 2))
    ax.axis('tight')
    ax.axis('off')
    
    table = ax.table(cellText=df_diff_detailed.values, 
                     colLabels=df_diff_detailed.columns,
                     cellLoc='center', 
                     loc='center',
                     bbox=[0, 0, 1, 1])
    
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 2)
    
    # Estilizar cabeçalho
    for i in range(len(df_diff_detailed.columns)):
        table[(0, i)].set_facecolor('#FF8C00')
        table[(0, i)].set_text_props(weight='bold', color='white')
    
    # Estilizar células com cores baseadas em positivo/negativo
    for i in range(1, len(df_diff_detailed) + 1):
        for j in range(len(df_diff_detailed.columns)):
            if j > 0:  # Pular coluna do modelo
                cell_val = str(df_diff_detailed.iloc[i-1, j])
                if "+" in cell_val:
                    table[(i, j)].set_facecolor('#90EE90')  # Verde claro
                elif "-" in cell_val and "N/A" not in cell_val:
                    table[(i, j)].set_facecolor('#FFB6C6')  # Vermelho claro
    
    plt.savefig(diff_png_file, dpi=300, bbox_inches='tight', pad_inches=0.2)
    plt.close()
    print(f"✅ Tabela de Diferença exportada com sucesso (método alternativo)!")

print(f"   📄 LaTeX: {diff_latex_file}")
print(f"   🖼️  PNG: {diff_png_file}")
print("-"*100)

⚠️ Usando método alternativo para PNG: It looks like you are using Playwright Sync API inside the asyncio loop.
Please use the Async API instead.
✅ Tabela de Diferença exportada com sucesso (método alternativo)!
   📄 LaTeX: inference/llm_judge_result/exports/tabela_diff_mti_sti_llm_judge.tex
   🖼️  PNG: inference/llm_judge_result/exports/tabela_diff_mti_sti_llm_judge.png
----------------------------------------------------------------------------------------------------
✅ Tabela de Diferença exportada com sucesso (método alternativo)!
   📄 LaTeX: inference/llm_judge_result/exports/tabela_diff_mti_sti_llm_judge.tex
   🖼️  PNG: inference/llm_judge_result/exports/tabela_diff_mti_sti_llm_judge.png
----------------------------------------------------------------------------------------------------


In [ ]:
print("\n" + "="*100)
print("📦 RESUMO DA EXPORTAÇÃO")
print("="*100)

print(f"\n✅ Arquivos exportados com sucesso para o diretório: {export_dir}\n")

print("📊 TABELA MTI (Multi-Task Inference):")
print(f"   • Arquivo LaTeX: {mti_latex_file}")
print(f"   • Arquivo PNG: {mti_png_file}")

print("\n📊 TABELA STI (Single-Task Inference):")
print(f"   • Arquivo LaTeX: {sti_latex_file}")
print(f"   • Arquivo PNG: {sti_png_file}")

print("\n📊 TABELA DE DIFERENÇA (Δ = MTI - STI):")
print(f"   • Arquivo LaTeX: {diff_latex_file}")
print(f"   • Arquivo PNG: {diff_png_file}")

print("\n" + "="*100)
print("📝 INSTRUÇÕES DE USO NO LATEX")
print("="*100)

print("""
Para usar as tabelas no seu documento LaTeX:

1️⃣ PACOTES NECESSÁRIOS:
   
   No preâmbulo do seu documento, adicione:
   \\usepackage{booktabs}
   \\usepackage{adjustbox}
   \\usepackage{makecell}
   \\usepackage{xcolor}  % Para cores na tabela de diferença
   
2️⃣ INSERIR AS TABELAS:
   
   No corpo do documento, onde deseja inserir as tabelas:
   \\input{path/to/tabela_mti_llm_judge.tex}
   \\input{path/to/tabela_sti_llm_judge.tex}
   \\input{path/to/tabela_diff_mti_sti_llm_judge.tex}

3️⃣ FORMATO DAS TABELAS MTI/STI:
   • Max: [%] - Porcentagem de respostas com pontuação máxima (5)
   • μ: [Média] - Pontuação média na escala Likert (1-5)
   • σ: [Desvio Padrão] - Variabilidade das respostas
   
4️⃣ FORMATO DA TABELA DE DIFERENÇA:
   • Valores em VERDE (positivos): MTI superior à STI
   • Valores em VERMELHO (negativos): STI superior à MTI
   • ΔMax: Diferença percentual de notas máximas
   • Δμ: Diferença nas médias
   • Δσ: Diferença nos desvios padrão
   
5️⃣ DICAS:
   • As tabelas já incluem caption e label
   • Para formato paisagem: \\usepackage{rotating} e \\begin{sidewaystable}
   • Para ajustar fonte: \\small ou \\footnotesize antes da tabela
   • Referências: \\ref{tab:mti_llm_judge}, \\ref{tab:sti_llm_judge}, \\ref{tab:diff_mti_sti_llm_judge}
""")

print("="*100)
print(f"🎉 Exportação concluída com sucesso!")
print("="*100)

In [ ]:
"""
    Código tex que deve ser gerado
    %%%% Resultado LLM Judge (PIPELINE FINAL) MTI %%%%
    \begin{table}[htbp]
    \caption{Resultados Detalhados da Abordagem MTI - Métricas LLM Judge por Modelo}
    \label{tab:mti_llm_judge}
    \centering

    \renewcommand{\arraystretch}{1.45} % aumenta o espaçamento entre linhas

    \begin{adjustbox}{max width=\textwidth}
    \begin{tabular}{ccccccc}
    \toprule
    \textbf{Modelo} & \textbf{Coerência} & \textbf{Especificidade} & \textbf{Informatividade} & \textbf{Relevância} & \textbf{Compreensibilidade} & \textbf{Média} \\
    \midrule

    \makecell{llama2:7b} &
    \makecell[c]{Max: 14.92\% \\ $\mu$: 3.84 \\ $\sigma$: 0.71} &
    \makecell[c]{Max: 11.83\% \\ $\mu$: 3.59 \\ $\sigma$: 0.95} &
    \makecell[c]{Max: 17.29\% \\ $\mu$: 3.82 \\ $\sigma$: 0.78} &
    \makecell[c]{Max: 67.79\% \\ $\mu$: 4.56 \\ $\sigma$: 0.74} &
    \makecell[c]{Max: 25.04\% \\ $\mu$: 3.96 \\ $\sigma$: 0.78} &
    \makecell[c]{Max: 27.38\% \\ $\mu$: 3.95 \\ $\sigma$: 0.86} \\
    \midrule

    \makecell{gpt-3.5-turbo-0125} &
    \makecell[c]{Max: 45.88\% \\ $\mu$: 4.34 \\ $\sigma$: 0.74} &
    \makecell[c]{Max: 35.79\% \\ $\mu$: 3.96 \\ $\sigma$: 1.15} &
    \makecell[c]{Max: 39.38\% \\ $\mu$: 4.14 \\ $\sigma$: 0.89} &
    \makecell[c]{Max: 84.96\% \\ $\mu$: 4.79 \\ $\sigma$: 0.60} &
    \makecell[c]{Max: 55.50\% \\ $\mu$: 4.47 \\ $\sigma$: 0.71} &
    \makecell[c]{Max: 52.30\% \\ $\mu$: 4.34 \\ $\sigma$: 0.89} \\
    \midrule

    \makecell{llama-3.3-70b-versatile} &
    \makecell[c]{Max: 53.87\% \\ $\mu$: 4.52 \\ $\sigma$: 0.53} &
    \makecell[c]{Max: 44.50\% \\ $\mu$: 4.28 \\ $\sigma$: 0.94} &
    \makecell[c]{Max: 49.54\% \\ $\mu$: 4.46 \\ $\sigma$: 0.57} &
    \makecell[c]{Max: 95.67\% \\ $\mu$: 4.96 \\ $\sigma$: 0.21} &
    \makecell[c]{Max: 65.96\% \\ $\mu$: 4.65 \\ $\sigma$: 0.49} &
    \makecell[c]{Max: 61.91\% \\ $\mu$: 4.57 \\ $\sigma$: 0.64} \\
    \midrule

    \makecell{gpt-4o-mini-2024-07-18} &
    \makecell[c]{Max: 63.92\% \\ $\mu$: 4.63 \\ $\sigma$: 0.50} &
    \makecell[c]{Max: 53.08\% \\ $\mu$: 4.37 \\ $\sigma$: 0.96} &
    \makecell[c]{Max: 57.67\% \\ $\mu$: 4.55 \\ $\sigma$: 0.55} &
    \makecell[c]{Max: 95.96\% \\ $\mu$: 4.96 \\ $\sigma$: 0.20} &
    \makecell[c]{Max: 71.67\% \\ $\mu$: 4.71 \\ $\sigma$: 0.47} &
    \makecell[c]{Max: 68.46\% \\ $\mu$: 4.64 \\ $\sigma$: 0.62} \\
    \bottomrule

    \end{tabular}
    \end{adjustbox}

    \end{table}




    %%%% Resultado LLM Judge (PIPELINE FINAL) STI %%%%
    \begin{table}[htbp]
    \caption{Resultados Detalhados da Abordagem STI - Métricas LLM Judge por Modelo. 
    Porcentagem de Nota Máxima (Max); Média ($\mu$); Desvio Padrão ($\sigma$)}
    \label{tab:sti_llm_judge}
    \centering

    \renewcommand{\arraystretch}{1.45}

    \begin{adjustbox}{max width=\textwidth}
    \begin{tabular}{ccccccc}
    \toprule
    \textbf{Modelo} & \textbf{Coerência} & \textbf{Especificidade} & \textbf{Informatividade} & \textbf{Relevância} & \textbf{Compreensibilidade} & \textbf{Média} \\
    \midrule

    \makecell{llama2:7b} &
    \makecell[c]{Max: 7.92\% \\ $\mu$: 3.56 \\ $\sigma$: 0.76} &
    \makecell[c]{Max: 6.67\% \\ $\mu$: 3.31 \\ $\sigma$: 0.98} &
    \makecell[c]{Max: 10.58\% \\ $\mu$: 3.55 \\ $\sigma$: 0.85} &
    \makecell[c]{Max: 51.33\% \\ $\mu$: 4.28 \\ $\sigma$: 0.90} &
    \makecell[c]{Max: 15.04\% \\ $\mu$: 3.68 \\ $\sigma$: 0.82} &
    \makecell[c]{Max: 18.31\% \\ $\mu$: 3.68 \\ $\sigma$: 0.92} \\
    \midrule

    \makecell{gpt-3.5-turbo-0125} &
    \makecell[c]{Max: 47.33\% \\ $\mu$: 4.40 \\ $\sigma$: 0.63} &
    \makecell[c]{Max: 37.50\% \\ $\mu$: 4.08 \\ $\sigma$: 1.07} &
    \makecell[c]{Max: 42.71\% \\ $\mu$: 4.29 \\ $\sigma$: 0.74} &
    \makecell[c]{Max: 89.12\% \\ $\mu$: 4.87 \\ $\sigma$: 0.40} &
    \makecell[c]{Max: 57.42\% \\ $\mu$: 4.53 \\ $\sigma$: 0.59} &
    \makecell[c]{Max: 54.82\% \\ $\mu$: 4.43 \\ $\sigma$: 0.77} \\
    \midrule

    \makecell{llama-3.3-70b-versatile} &
    \makecell[c]{Max: 45.88\% \\ $\mu$: 4.41 \\ $\sigma$: 0.58} &
    \makecell[c]{Max: 37.83\% \\ $\mu$: 4.19 \\ $\sigma$: 0.94} &
    \makecell[c]{Max: 42.42\% \\ $\mu$: 4.37 \\ $\sigma$: 0.59} &
    \makecell[c]{Max: 91.83\% \\ $\mu$: 4.91 \\ $\sigma$: 0.34} &
    \makecell[c]{Max: 57.42\% \\ $\mu$: 4.55 \\ $\sigma$: 0.55} &
    \makecell[c]{Max: 55.07\% \\ $\mu$: 4.49 \\ $\sigma$: 0.67} \\
    \midrule

    \makecell{gpt-4o-mini-2024-07-18} &
    \makecell[c]{Max: 57.71\% \\ $\mu$: 4.57 \\ $\sigma$: 0.52} &
    \makecell[c]{Max: 48.21\% \\ $\mu$: 4.33 \\ $\sigma$: 0.95} &
    \makecell[c]{Max: 53.17\% \\ $\mu$: 4.51 \\ $\sigma$: 0.54} &
    \makecell[c]{Max: 96.46\% \\ $\mu$: 4.96 \\ $\sigma$: 0.19} &
    \makecell[c]{Max: 65.21\% \\ $\mu$: 4.64 \\ $\sigma$: 0.50} &
    \makecell[c]{Max: 64.15\% \\ $\mu$: 4.60 \\ $\sigma$: 0.62} \\
    \bottomrule

    \end{tabular}
    \end{adjustbox}

    \end{table}


"""

